In [1]:
# Celda 1: Instalación de librerías
!pip install langchain langgraph langchain-groq -q
print("Librerías instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00
Librerías instaladas correctamente


In [2]:
# Celda 2: Configuración de API Key
import os
from google.colab import userdata

# Configuramos la API Key de Groq
os.environ["GROQ_API_KEY"] = "PEGA_TU_API_KEY_AQUI"

print("API Key configurada correctamente")

API Key configurada correctamente


In [4]:
# Celda 3: Conexión con Groq y prueba del modelo
from langchain_groq import ChatGroq

# Usamos gpt-oss-20b disponible en tu cuenta
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

# Prueba simple
respuesta = llm.invoke("Hola, ¿puedes responder en español?")
print("✅ Conexión exitosa con Groq")
print(f"Respuesta del modelo: {respuesta.content}")

✅ Conexión exitosa con Groq
Respuesta del modelo: ¡Claro! Puedo responder en español. ¿En qué puedo ayudarte hoy?


In [5]:
# Celda 4: Agente ReAct para consultas bancarias
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import create_react_agent

# --- HERRAMIENTAS DEL AGENTE ---

@tool
def consultar_politica_bancaria(categoria: str) -> str:
    """Consulta la política bancaria según la categoría de la consulta."""
    politicas = {
        "bloqueo_tarjeta": "Para bloquear una tarjeta: verificar identidad con cédula + OTP. El bloqueo es inmediato. Se genera número de caso. Reposición en 3-5 días hábiles.",
        "desbloqueo_tarjeta": "Para desbloquear: verificar que el bloqueo fue voluntario. Si fue por intentos fallidos de PIN, derivar a sucursal.",
        "credito_consumo": "Requisitos crédito consumo: cédula, 3 roles de pago, planilla servicios básicos, score mínimo 650. Tasa 16.5% anual. Cuota máxima 40% del ingreso.",
        "certificado_cuenta": "Certificado de cuenta activa: costo $3.00, emisión inmediata en app/web o 1 hora en sucursal. Incluye saldo promedio 6 meses.",
        "reseteo_clave": "Reseteo clave banca web: opción 'Olvidé mi clave' en login, código OTP al email o teléfono registrado. Nueva clave mínimo 8 caracteres.",
        "desgravamen": "Seguro desgravamen: se da de baja automáticamente al cancelar el crédito. Si hay primas no devengadas, se reembolsan en 5-7 días hábiles.",
    }
    return politicas.get(categoria, "Categoría no encontrada. Categorías disponibles: " + ", ".join(politicas.keys()))

@tool
def verificar_identidad(cedula: str) -> str:
    """Verifica la identidad del cliente con su número de cédula."""
    # Simulación — en producción consultaría la base de datos real
    if len(cedula) == 10 and cedula.isdigit():
        return f"Identidad verificada para cédula {cedula[:3]}*******. Cliente activo."
    return "❌ Número de cédula inválido. Debe tener 10 dígitos."

@tool
def generar_numero_caso(tipo: str) -> str:
    """Genera un número de caso para el seguimiento de la solicitud."""
    import random
    codigo = random.randint(1000, 9999)
    prefijos = {"bloqueo": "BC", "fraude": "FR", "credito": "CR", "certificado": "CERT", "clave": "CL"}
    prefijo = prefijos.get(tipo.lower(), "GEN")
    return f"Número de caso generado: #{prefijo}-2026-{codigo}"

# --- CREAR EL AGENTE REACT ---
herramientas = [consultar_politica_bancaria, verificar_identidad, generar_numero_caso]

agente = create_react_agent(llm, herramientas)

print("Agente ReAct creado con 3 herramientas bancarias")
print("Herramientas disponibles:", [h.name for h in herramientas])

✅ Agente ReAct creado con 3 herramientas bancarias
Herramientas disponibles: ['consultar_politica_bancaria', 'verificar_identidad', 'generar_numero_caso']


/tmp/ipykernel_4020/2532712836.py:41: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(llm, herramientas)


In [6]:
# Celda 5: Prueba del agente con consulta bancaria real
from langchain_core.messages import HumanMessage

# Sistema de instrucciones del agente
system_prompt = """Eres un agente de soporte bancario inteligente del Banco Ecuador.
Tu objetivo es resolver consultas de clientes usando las herramientas disponibles.
Debes:
1. Analizar la consulta del cliente
2. Usar las herramientas necesarias para obtener información
3. Dar una respuesta completa y precisa
4. Generar un número de caso cuando sea necesario
Responde siempre en español de forma profesional y clara."""

# Consulta de prueba — Escenario E1.1 (bloqueo de tarjeta)
consulta = """
Cliente: María Torres, cédula 1723456789
Consulta: Necesito bloquear mi tarjeta de débito, la perdí esta mañana en el transporte.
"""

print("="*60)
print("CONSULTA DEL CLIENTE:")
print(consulta)
print("="*60)
print("\nPROCESAMIENTO DEL AGENTE ReAct:\n")

# Ejecutar el agente
resultado = agente.invoke({
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": consulta}
    ]
})

# Mostrar el razonamiento paso a paso
pasos = 0
for mensaje in resultado["messages"]:
    tipo = type(mensaje).__name__
    if "ToolMessage" in tipo:
        pasos += 1
        print(f"🔧 Herramienta usada: {mensaje.name}")
        print(f"   Resultado: {mensaje.content[:100]}...")
        print()

print("="*60)
print("RESPUESTA FINAL DEL AGENTE:")
print(resultado["messages"][-1].content)
print("="*60)
print(f"\n📊 MÉTRICA: Pasos de razonamiento utilizados: {pasos}")

CONSULTA DEL CLIENTE:

Cliente: María Torres, cédula 1723456789
Consulta: Necesito bloquear mi tarjeta de débito, la perdí esta mañana en el transporte.


PROCESAMIENTO DEL AGENTE ReAct:

🔧 Herramienta usada: verificar_identidad
   Resultado: ✅ Identidad verificada para cédula 172*******. Cliente activo....

🔧 Herramienta usada: consultar_politica_bancaria
   Resultado: Categoría no encontrada. Categorías disponibles: bloqueo_tarjeta, desbloqueo_tarjeta, credito_consum...

🔧 Herramienta usada: consultar_politica_bancaria
   Resultado: Para bloquear una tarjeta: verificar identidad con cédula + OTP. El bloqueo es inmediato. Se genera ...

🔧 Herramienta usada: generar_numero_caso
   Resultado: Número de caso generado: #GEN-2026-9251...

RESPUESTA FINAL DEL AGENTE:
Estimada María Torres,

Hemos verificado su identidad con la cédula **1723456789** y procedido a bloquear su tarjeta de débito de inmediato.  
A continuación, le indicamos los pasos a seguir:

1. **OTP (Código de Verificación)*